# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their unique `@id`. This ensures reproducible, precise dataset interaction and is aligned with the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. Restart the kernel afterwards if necessary.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets in the dataset. All entities are referenced by their `@id`, as per Croissant specification.

In [ ]:
# Explore available record sets and their fields
print("Record Sets in the Dataset:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'dataType', None)}")
    print()

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames for analysis. 
Ensure to use the `@id` of the record set and fields as found above.

In [ ]:
# Select all available record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"Available record set @ids: {record_set_ids}")

# Extract the first record set as an example
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nLoading records from record set: {selected_record_set_id}")
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records. Columns (field @ids):")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering and normalization, referring to all fields by `@id` as returned above.

In [ ]:
import numpy as np

# Let's inspect numeric fields by checking the dataType in the first record set
numeric_field_ids = []
group_field_id = None
if record_set_ids:
    record_set = [r for r in metadata.record_sets if r.id == selected_record_set_id][0]
    for field in record_set.fields:
        dt = getattr(field, 'dataType', '').lower() if getattr(field, 'dataType', None) else ''
        if dt in ('float', 'integer', 'number'):
            numeric_field_ids.append(field.id)
        # Use the first field with string/text type as group field
        if group_field_id is None and (dt in (None, '', 'string', 'text')):
            group_field_id = field.id

print(f"Numeric fields (@id): {numeric_field_ids}")
print(f"Group (categorical) field candidate (@id): {group_field_id}\n")

if numeric_field_ids:
    # For demonstration, pick the first numeric field
    numeric_field = numeric_field_ids[0]
    # Choose a threshold (e.g., 10)
    threshold = 10
    if numeric_field in df:
        # Filter where the value is greater than the threshold
        numeric_ser = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df.loc[numeric_ser > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (using field @id): {len(filtered_df)} found.")
        
        # Normalize the numeric field
        mean = numeric_ser.mean()
        std = numeric_ser.std()
        filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - mean) / std
        print(f"Normalized {numeric_field} (field @id):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by group_field_id, if available
        if group_field_id and group_field_id in df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field].mean()
            print(f"Mean of {numeric_field} grouped by {group_field_id} (all references by @id):")
            display(grouped.head())
        else:
            print('No suitable group field available for grouping.')
    else:
        print(f"Field {numeric_field} not found in the DataFrame columns.")
else:
    print("No numeric fields found in this record set for EDA.")

## 5. Visualization
Visualize data distributions for numeric fields using matplotlib or seaborn, again referencing by `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the first numeric field
if numeric_field_ids and numeric_field in df:
    data = pd.to_numeric(df[numeric_field], errors='coerce').dropna()
    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field, if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field_id, grid=False)
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a FAIR² Croissant dataset with `mlcroissant`
- Identify and reference all main entities (record sets, fields) by their `@id`
- Extract tabular data for processing and analysis
- Conduct basic exploratory data analysis and visualize results, maintaining strict referencing by `@id` as per Croissant best practices.

Use these methods as a foundation for deeper statistical analyses and reproducible workflows across standardized Croissant datasets.